In [6]:
using Pkg                # Load Julia's built-in package manager module (Pkg)

Pkg.activate("..")       # Activate the Project.toml located one directory above
                         # (Allows notebooks/ to use the repository root environment)

using Horizons3DViewer2  # Load the custom package Horizons3DViewer2
                         # (This loads src/Horizons3DViewer2.jl and its modules)

using EphemerisSources   # Tools for accessing planetary ephemeris data
using AstroTime, Dates   # Time systems and date handling utilities
using GLMakie            # High-performance interactive visualization backend
using ProgressMeter      # Progress bar utilities for long computations
using LinearAlgebra      # Standard linear algebra operations (norm, dot, etc.)
using SPICE

  Activating project at `~/work/Horizons3DViewer2`


In [7]:
using GLMakie
using LinearAlgebra
using Printf

# ============================================================
# 定数
# ============================================================

const C = 299_792_458.0   # 光速 [m/s]
const F_TX = 2.2e9        # Sバンド送信周波数 [Hz]


# ============================================================
# 送信物体の真の状態
#
# 位置は描画用に km
# 速度は m/s
# ============================================================



# ============================================================
# 補助関数
# ============================================================

"""
    point_to_vector(p)

Point2fを通常のFloat64ベクトルへ変換する。
"""
point_to_vector(p) = Float64[p[1], p[2]]


"""
    line_of_sight(object_position, station_position)

受信局から物体へ向かう単位視線ベクトルを計算する。
"""
function line_of_sight(object_position, station_position)
    relative_position =
        point_to_vector(object_position) -
        point_to_vector(station_position)

    return relative_position / norm(relative_position)
end


"""
    range_km(object_position, station_position)

物体と受信局の距離 [km] を計算する。
"""
function range_km(object_position, station_position)
    dx = object_position[1] - station_position[1]
    dy = object_position[2] - station_position[2]

    return hypot(dx, dy)
end


"""
    calculate_measurements(...)

3局について、局1を基準とするTDOAとFDOAを計算する。

FDOAの定義:

    FDOA₂₁ = f_rx,2 - f_rx,1
    FDOA₃₁ = f_rx,3 - f_rx,1
"""
function calculate_measurements(
    object_position,
    object_velocity,
    station1,
    station2,
    station3;
    f_tx = F_TX,
    c = C,
)
    stations = [station1, station2, station3]

    ranges_m = zeros(3)
    range_rates_ms = zeros(3)
    received_frequencies_hz = zeros(3)

    for i in eachindex(stations)
        ranges_m[i] =
            range_km(object_position, stations[i]) * 1000.0

        los = line_of_sight(
            object_position,
            stations[i],
        )

        # 受信局は静止していると仮定
        range_rates_ms[i] = dot(
            los,
            object_velocity,
        )

        # 一次ドップラー近似
        received_frequencies_hz[i] =
            f_tx * (1.0 - range_rates_ms[i] / c)
    end

    # 局1を基準としたTDOA
    tdoa21_s = (ranges_m[2] - ranges_m[1]) / c
    tdoa31_s = (ranges_m[3] - ranges_m[1]) / c

    # 局1を基準としたFDOA
    fdoa21_hz =
        received_frequencies_hz[2] -
        received_frequencies_hz[1]

    fdoa31_hz =
        received_frequencies_hz[3] -
        received_frequencies_hz[1]

    return (
        ranges_m = ranges_m,
        range_rates_ms = range_rates_ms,
        received_frequencies_hz = received_frequencies_hz,
        tdoa21_s = tdoa21_s,
        tdoa31_s = tdoa31_s,
        fdoa21_hz = fdoa21_hz,
        fdoa31_hz = fdoa31_hz,
    )
end


"""
    estimate_velocity_from_fdoa(...)

物体位置が既知であるとして、2個のFDOAから
2次元速度 [vx, vy] を推定する。

式:

    (u₂ - u₁)'v = -(c/f₀) FDOA₂₁
    (u₃ - u₁)'v = -(c/f₀) FDOA₃₁
"""
function estimate_velocity_from_fdoa(
    object_position,
    station1,
    station2,
    station3,
    fdoa21_hz,
    fdoa31_hz;
    f_tx = F_TX,
    c = C,
    determinant_tolerance = 1e-6,
)
    u1 = line_of_sight(object_position, station1)
    u2 = line_of_sight(object_position, station2)
    u3 = line_of_sight(object_position, station3)

    du21 = u2 - u1
    du31 = u3 - u1

    A = [
        du21[1] du21[2]
        du31[1] du31[2]
    ]

    b = -c / f_tx .* [
        fdoa21_hz,
        fdoa31_hz,
    ]

    determinant = det(A)

    if abs(determinant) < determinant_tolerance
        return (
            velocity_ms = [NaN, NaN],
            determinant = determinant,
            condition_number = Inf,
            valid = false,
        )
    end

    velocity_ms = A \ b

    return (
        velocity_ms = velocity_ms,
        determinant = determinant,
        condition_number = cond(A),
        valid = true,
    )
end


# ============================================================
# Figure
# ============================================================

fig = Figure(
    size = (2000, 1000),
)

# 左：位置平面
ax_position = Axis(
    fig[1:2, 1:2],
    xlabel = "x [km]",
    ylabel = "y [km]",
    title = "Position plane: TDOA hyperbolas",
    aspect = DataAspect(),
)

xlims!(ax_position, -400, 400)
ylims!(ax_position, -350, 400)

# 中央：速度平面
ax_velocity = Axis(
    fig[1:2, 3:4],
    xlabel = "vx [m/s]",
    ylabel = "vy [m/s]",
    title = "Velocity plane: FDOA constraint lines",
    aspect = DataAspect(),
)

xlims!(ax_velocity, -250, 250)
ylims!(ax_velocity, -250, 250)


# ============================================================
# 受信局位置スライダー
# ============================================================

slider_grid = SliderGrid(
    fig[4, 1:2],

    (
        label = "Station 1: x [km]",
        range = -350.0:5.0:350.0,
        startvalue = -200.0,
        format = value -> @sprintf("%.0f", value),
    ),

    (
        label = "Station 1: y [km]",
        range = -300.0:5.0:350.0,
        startvalue = -100.0,
        format = value -> @sprintf("%.0f", value),
    ),

    (
        label = "Station 2: x [km]",
        range = -350.0:5.0:350.0,
        startvalue = 200.0,
        format = value -> @sprintf("%.0f", value),
    ),

    (
        label = "Station 2: y [km]",
        range = -300.0:5.0:350.0,
        startvalue = -100.0,
        format = value -> @sprintf("%.0f", value),
    ),

    (
        label = "Station 3: x [km]",
        range = -350.0:5.0:350.0,
        startvalue = 0.0,
        format = value -> @sprintf("%.0f", value),
    ),

    (
        label = "Station 3: y [km]",
        range = -300.0:5.0:350.0,
        startvalue = 300.0,
        format = value -> @sprintf("%.0f", value),
    ),

    width = 430,
)

slider_values = [
    slider.value
    for slider in slider_grid.sliders
]


# ============================================================
# 送信物体の位置・速度スライダー
# 速度は極座標（speed, angle）で指定する。
# ============================================================

object_slider_grid = SliderGrid(
    fig[4, 3:4],

    (
        label = "Object: x [km]",
        range = -400.0:5.0:400.0,
        startvalue = 120.0,
        format = value -> @sprintf("%.0f", value),
    ),

    (
        label = "Object: y [km]",
        range = -350.0:5.0:400.0,
        startvalue = 180.0,
        format = value -> @sprintf("%.0f", value),
    ),

    (
        label = "Object: speed [m/s]",
        range = 0.0:5.0:300.0,
        startvalue = 126.0,
        format = value -> @sprintf("%.0f", value),
    ),

    (
        label = "Object: angle [deg]",
        range = -180.0:1.0:180.0,
        startvalue = -18.0,
        format = value -> @sprintf("%.0f", value),
    ),

    width = 430,
)

object_slider_values = [
    slider.value
    for slider in object_slider_grid.sliders
]


# ============================================================
# スライダー値から各局位置を生成
# ============================================================

station1 = lift(
    slider_values[1],
    slider_values[2],
) do x, y
    Point2f(x, y)
end

station2 = lift(
    slider_values[3],
    slider_values[4],
) do x, y
    Point2f(x, y)
end

station3 = lift(
    slider_values[5],
    slider_values[6],
) do x, y
    Point2f(x, y)
end

object_position_km = lift(
    object_slider_values[1],
    object_slider_values[2],
) do x, y
    Point2f(x, y)
end

true_velocity_ms = lift(
    object_slider_values[3],
    object_slider_values[4],
) do speed, angle_deg
    [
        speed * cosd(angle_deg),
        speed * sind(angle_deg),
    ]
end

object_position_marker = lift(object_position_km) do position
    [position]
end

true_velocity_direction = lift(
    true_velocity_ms,
 ) do velocity
    [
        Vec2f(
            0.5 * velocity[1],
            0.5 * velocity[2],
        ),
    ]
end

true_velocity_point = lift(true_velocity_ms) do velocity
    [Point2f(velocity[1], velocity[2])]
end


# ============================================================
# TDOA・FDOA観測値
# ============================================================

measurements = lift(
    object_position_km,
    true_velocity_ms,
    station1,
    station2,
    station3,
) do position, velocity, s1, s2, s3

    calculate_measurements(
        position,
        velocity,
        s1,
        s2,
        s3,
    )
end


# ============================================================
# FDOAから速度推定
# ============================================================

velocity_solution = lift(
    object_position_km,
    station1,
    station2,
    station3,
    measurements,
) do position, s1, s2, s3, measurement

    estimate_velocity_from_fdoa(
        position,
        s1,
        s2,
        s3,
        measurement.fdoa21_hz,
        measurement.fdoa31_hz,
    )
end


# ============================================================
# TDOA双曲線用グリッド
#
# 双曲線を陽関数として書く代わりに、
#
#   distance(r, S₂) - distance(r, S₁) - Δρ₂₁ = 0
#
# のゼロ等高線を描く。
# ============================================================

x_grid = range(-400.0, 400.0, length = 401)
y_grid = range(-350.0, 400.0, length = 376)


# S2-S1の距離差
range_difference21_km = lift(
    object_position_km,
    station1,
    station2,
) do position, s1, s2

    range_km(position, s2) -
    range_km(position, s1)
end


# S3-S1の距離差
range_difference31_km = lift(
    object_position_km,
    station1,
    station3,
) do position, s1, s3

    range_km(position, s3) -
    range_km(position, s1)
end


# 双曲線21の陰関数
hyperbola21 = lift(
    station1,
    station2,
    range_difference21_km,
) do s1, s2, delta_range

    [
        hypot(x - s2[1], y - s2[2]) -
        hypot(x - s1[1], y - s1[2]) -
        delta_range

        for x in x_grid, y in y_grid
    ]
end


# 双曲線31の陰関数
hyperbola31 = lift(
    station1,
    station3,
    range_difference31_km,
) do s1, s3, delta_range

    [
        hypot(x - s3[1], y - s3[2]) -
        hypot(x - s1[1], y - s1[2]) -
        delta_range

        for x in x_grid, y in y_grid
    ]
end


# ============================================================
# 双曲線描画
# ============================================================

contour!(
    ax_position,
    x_grid,
    y_grid,
    hyperbola21;
    levels = [0.0],
    color = :cyan,
    linewidth = 3,
    label = "TDOA: Station 2 - Station 1",
)

contour!(
    ax_position,
    x_grid,
    y_grid,
    hyperbola31;
    levels = [0.0],
    color = :magenta,
    linewidth = 3,
    label = "TDOA: Station 3 - Station 1",
)


# ============================================================
# 受信局描画
# ============================================================

station_points = lift(
    station1,
    station2,
    station3,
) do s1, s2, s3
    [s1, s2, s3]
end

scatter!(
    ax_position,
    station_points;
    color = [:red, :green, :blue],
    markersize = 22,
)

station_labels = lift(
    station1,
    station2,
    station3,
) do s1, s2, s3
    [
        (position = s1, text = "S1"),
        (position = s2, text = "S2"),
        (position = s3, text = "S3"),
    ]
end

for i in 1:3
    label_position = lift(station_labels) do labels
        labels[i].position
    end

    label_text = lift(station_labels) do labels
        labels[i].text
    end

    text!(
        ax_position,
        label_position;
        text = label_text,
        offset = (12, 12),
        fontsize = 20,
        font = :bold,
    )
end


# ============================================================
# 物体位置
# ============================================================

scatter!(
    ax_position,
    object_position_marker;
    color = :yellow,
    strokecolor = :black,
    strokewidth = 2,
    markersize = 25,
    label = "Transmitter",
)

text!(
    ax_position,
    object_position_marker;
    text = "Object",
    offset = (12, 12),
    fontsize = 20,
    font = :bold,
)


# ============================================================
# 真の速度ベクトル
#
# 描画上の長さを調整するため、
# 1 m/sを0.5 kmとして表示する。
# ============================================================

velocity_plot_scale = 0.5   # 描画上の km / (m/s)

arrows2d!(
    ax_position,
    object_position_marker,
    true_velocity_direction;
    color = :lime,
    label = "True velocity",
)


# ============================================================
# FDOA推定速度ベクトル
# ============================================================

estimated_velocity_direction = lift(
    velocity_solution,
) do solution

    if solution.valid
        velocity = solution.velocity_ms

        [
            Vec2f(
                velocity_plot_scale * velocity[1],
                velocity_plot_scale * velocity[2],
            ),
        ]
    else
        [Vec2f(0.0, 0.0)]
    end
end

estimated_velocity_visible = lift(
    velocity_solution,
) do solution
    solution.valid
end

arrows2d!(
    ax_position,
    object_position_marker,
    estimated_velocity_direction;
    color = :orange,
    visible = estimated_velocity_visible,
    label = "Velocity estimated from FDOA",
)


# ============================================================
# 結果表示
# ============================================================

result_text = lift(
    object_position_km,
    true_velocity_ms,
    measurements,
    velocity_solution,
) do position, true_velocity, measurement, solution

    if solution.valid
        velocity = solution.velocity_ms

        return @sprintf(
            """
            Transmitted frequency
              f₀ = %.3f GHz

            Object state
              x = %.1f km
              y = %.1f km

            TDOA observations
              TDOA₂₁ = %+.6f μs
              TDOA₃₁ = %+.6f μs

            FDOA observations
              FDOA₂₁ = %+.6f Hz
              FDOA₃₁ = %+.6f Hz

            True velocity
              vx = %+.3f m/s
              vy = %+.3f m/s

            Estimated velocity
              vx = %+.3f m/s
              vy = %+.3f m/s

            Velocity geometry
              det(A) = %+.6e
              cond(A) = %.3f
            """,
            F_TX / 1e9,
            position[1],
            position[2],
            measurement.tdoa21_s * 1e6,
            measurement.tdoa31_s * 1e6,
            measurement.fdoa21_hz,
            measurement.fdoa31_hz,
            true_velocity[1],
            true_velocity[2],
            velocity[1],
            velocity[2],
            solution.determinant,
            solution.condition_number,
        )
    else
        return @sprintf(
            """
            Object state
              x = %.1f km
              y = %.1f km

            TDOA observations
              TDOA₂₁ = %+.6f μs
              TDOA₃₁ = %+.6f μs

            FDOA observations
              FDOA₂₁ = %+.6f Hz
              FDOA₃₁ = %+.6f Hz

            Velocity cannot be determined uniquely.

            det(A) = %+.6e

            Move the stations so that the two
            FDOA constraint lines are independent.
            """,
            position[1],
            position[2],
            measurement.tdoa21_s * 1e6,
            measurement.tdoa31_s * 1e6,
            measurement.fdoa21_hz,
            measurement.fdoa31_hz,
            solution.determinant,
        )
    end
end

Label(
    fig[1:4, 5],
    result_text;
    tellwidth = false,
    justification = :left,
    halign = :left,
    fontsize = 18,
)


# ============================================================
# FDOA制約直線
# ============================================================

vx_grid = range(-250.0, 250.0, length = 501)

fdoa_constraint_lines = lift(
    object_position_km,
    station1,
    station2,
    station3,
    measurements,
) do position, s1, s2, s3, measurement

    u1 = line_of_sight(position, s1)
    u2 = line_of_sight(position, s2)
    u3 = line_of_sight(position, s3)

    # FDOA制約:
    #
    # (u2-u1)'v = -(c/f0) FDOA21
    # (u3-u1)'v = -(c/f0) FDOA31

    a21 = u2 - u1
    a31 = u3 - u1

    b21 = -C / F_TX * measurement.fdoa21_hz
    b31 = -C / F_TX * measurement.fdoa31_hz

    # a_x vx + a_y vy = b
    #
    # 通常は vy = (b-a_x vx)/a_y として描画する。
    # a_y ≈ 0 の場合は鉛直線になるため、別処理する。

    vy21 = if abs(a21[2]) > 1e-10
        [
            (b21 - a21[1] * vx) / a21[2]
            for vx in vx_grid
        ]
    else
        fill(NaN, length(vx_grid))
    end

    vy31 = if abs(a31[2]) > 1e-10
        [
            (b31 - a31[1] * vx) / a31[2]
            for vx in vx_grid
        ]
    else
        fill(NaN, length(vx_grid))
    end

    vx21_vertical = if abs(a21[2]) <= 1e-10 &&
                       abs(a21[1]) > 1e-10
        b21 / a21[1]
    else
        NaN
    end

    vx31_vertical = if abs(a31[2]) <= 1e-10 &&
                       abs(a31[1]) > 1e-10
        b31 / a31[1]
    else
        NaN
    end

    return (
        vy21 = vy21,
        vy31 = vy31,
        vx21_vertical = vx21_vertical,
        vx31_vertical = vx31_vertical,
    )
end

# ============================================================
# FDOA制約直線の描画
# ============================================================

vy21 = lift(fdoa_constraint_lines) do data
    data.vy21
end

vy31 = lift(fdoa_constraint_lines) do data
    data.vy31
end

lines!(
    ax_velocity,
    vx_grid,
    vy21;
    color = :cyan,
    linewidth = 4,
    label = "FDOA: Station 2 - Station 1",
)

lines!(
    ax_velocity,
    vx_grid,
    vy31;
    color = :magenta,
    linewidth = 4,
    label = "FDOA: Station 3 - Station 1",
)

vertical21_visible = lift(fdoa_constraint_lines) do data
    isfinite(data.vx21_vertical)
end

vertical31_visible = lift(fdoa_constraint_lines) do data
    isfinite(data.vx31_vertical)
end

vertical21_position = lift(fdoa_constraint_lines) do data
    isfinite(data.vx21_vertical) ? [data.vx21_vertical] : [0.0]
end

vertical31_position = lift(fdoa_constraint_lines) do data
    isfinite(data.vx31_vertical) ? [data.vx31_vertical] : [0.0]
end

vlines!(
    ax_velocity,
    vertical21_position;
    color = :cyan,
    linewidth = 4,
    visible = vertical21_visible,
)

vlines!(
    ax_velocity,
    vertical31_position;
    color = :magenta,
    linewidth = 4,
    visible = vertical31_visible,
)

# ============================================================
# 真の速度
# ============================================================

scatter!(
    ax_velocity,
    true_velocity_point;
    color = :lime,
    strokecolor = :black,
    strokewidth = 2,
    markersize = 24,
    label = "True velocity",
)

# ============================================================
# FDOAから推定した速度
# ============================================================

estimated_velocity_point = lift(
    velocity_solution,
) do solution

    if solution.valid
        return [
            Point2f(
                solution.velocity_ms[1],
                solution.velocity_ms[2],
            ),
        ]
    end

    return [Point2f(NaN, NaN)]
end

estimated_velocity_visible = lift(
    velocity_solution,
) do solution
    solution.valid
end

scatter!(
    ax_velocity,
    estimated_velocity_point;
    color = :orange,
    strokecolor = :black,
    strokewidth = 2,
    marker = :star5,
    markersize = 30,
    visible = estimated_velocity_visible,
    label = "Estimated velocity",
)


# ============================================================
# 原点から推定速度までのベクトル
# ============================================================

estimated_velocity_vector = lift(
    velocity_solution,
) do solution

    if solution.valid
        return [
            Vec2f(
                solution.velocity_ms[1],
                solution.velocity_ms[2],
            ),
        ]
    end

    return [Vec2f(0.0, 0.0)]
end

arrows2d!(
    ax_velocity,
    [Point2f(0.0, 0.0)],
    estimated_velocity_vector;
    color = :orange,
    #linewidth = 3,
    tipwidth = 12,
    tiplength = 16,
    visible = estimated_velocity_visible,
)

# ============================================================
# 推定速度ラベル
# ============================================================

estimated_velocity_label = lift(
    velocity_solution,
) do solution

    if solution.valid
        vx, vy = solution.velocity_ms

        return @sprintf(
            "intersection\n(%.1f, %.1f) m/s",
            vx,
            vy,
        )
    end

    return "No unique intersection"
end

estimated_velocity_label_position = lift(
    velocity_solution,
) do solution

    if solution.valid
        return Point2f(
            solution.velocity_ms[1],
            solution.velocity_ms[2],
        )
    end

    return Point2f(0.0, 0.0)
end

text!(
    ax_velocity,
    estimated_velocity_label_position;
    text = estimated_velocity_label,
    offset = (15, 15),
    fontsize = 18,
    visible = estimated_velocity_visible,
)

# ============================================================
# 速度平面の補助線
# ============================================================

vlines!(
    ax_velocity,
    [0.0];
    color = (:gray, 0.5),
    linewidth = 1,
)

hlines!(
    ax_velocity,
    [0.0];
    color = (:gray, 0.5),
    linewidth = 1,
)


Legend(
    fig[3, 1:2],
    ax_position;
    fontsize = 6,    
    #orientation = :horizontal,
    #tellwidth = false,
)


Legend(
    fig[3, 3:4],
    ax_velocity;
    fontsize = 6,
    #orientation = :horizontal,
    #tellwidth = false,
)

display(fig)

GLMakie.Screen(...)